# FIR Document Image Preprocessing: review notebook

All processing code lives in **`image_processing.py`** (OpenCV + NumPy only, runs locally). This notebook runs the same functions **step by step** and shows tables and before/after views, so each step can be reviewed and tuned.

For production use, call the module directly:
```python
from image_processing import DocumentPreprocessor
pre = DocumentPreprocessor()
finals = pre.process("doc.pdf", save=True, output_dir="out")   # list of final page images, in page order
```

## Step 0: Setup & configuration
- Settings: `CONFIG` in `image_processing.py` (all sizes in mm). Override any value here, e.g. `CONFIG["deskew"]["max_angle"] = 8`.
- Device: detected by `device.py` (Mac M5 / NVIDIA GB10 / CPU).
- Input: choose the PDFs and pages to review below.

In [ ]:
import json
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pypdfium2 as pdfium

import image_processing as ip
from image_processing import *          # step functions, CONFIG, PROFILE, helpers (same objects as the module)

%config InlineBackend.figure_format = 'retina'

In [ ]:
# What to review (processing settings: CONFIG in image_processing.py; override single values here if needed)
CONFIG["input"].update({
    "pdf_dir": "sample-docs",
    "pdf_files": ["FIR-2-2009.pdf"],    # None = every PDF in pdf_dir, or a list such as ["FIR-2-2009.pdf"]
    "page_range": None,                  # None = all pages, or (first, last) 1-based inclusive, e.g. (1, 3)
})
OUTPUT_DIR = Path("output")              # Step 11 saves {pdf}_{page}_original / _final.png / _applied.json here

In [ ]:
print_profile(PROFILE)                   # detected when image_processing was imported

In [ ]:
def show_page_grid(records: list[dict], image_key: str, title: str, cols: int = 8, thumb_h: int = 320) -> None:
    """Thumbnail grid of one image stage; flagged pages get a red title."""
    if not records:
        return
    rows = (len(records) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.0, rows * 2.9), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, rec in zip(axes.flat, records):
        img = rec["images"].get(image_key)
        if img is None:
            continue
        h, w = img.shape[:2]
        thumb = cv2.resize(img, (max(1, int(w * thumb_h / h)), thumb_h), interpolation=cv2.INTER_AREA)
        ax.imshow(cv2.cvtColor(thumb, cv2.COLOR_BGR2RGB) if thumb.ndim == 3 else thumb, cmap="gray")
        flagged = bool(rec["flags"])
        ax.set_title(f"p{rec['page_no']}" + (" ⚑" if flagged else ""), fontsize=9,
                     color="red" if flagged else "black")
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


# ---------- Discover input PDFs and create page records ----------

pdf_dir = Path(CONFIG["input"]["pdf_dir"])
if CONFIG["input"]["pdf_files"]:
    pdf_paths = [pdf_dir / name for name in CONFIG["input"]["pdf_files"]]
else:
    pdf_paths = sorted(pdf_dir.glob("*.pdf"))

missing = [str(p) for p in pdf_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"PDF(s) not found: {missing}")
if not pdf_paths:
    raise FileNotFoundError(f"No PDFs found in {pdf_dir.resolve()}")


page_records: list[dict] = []
print(f"{'Document':<22}{'Pages':>6}{'Selected':>10}{'Portrait':>10}{'Landscape':>11}")
for pdf_path in pdf_paths:
    doc = pdfium.PdfDocument(str(pdf_path))
    n_pages = len(doc)
    first, last = CONFIG["input"]["page_range"] or (1, n_pages)
    selected = range(max(1, first) - 1, min(last, n_pages))
    doc_records = [new_page_record(pdf_path.stem, pdf_path, i, doc[i], n_pages) for i in selected]
    doc.close()
    page_records.extend(doc_records)
    n_land = sum(r["pdf_orientation"] == "landscape" for r in doc_records)
    print(f"{pdf_path.stem:<22}{n_pages:>6}{len(doc_records):>10}{len(doc_records) - n_land:>10}{n_land:>11}")

print(f"\nStep 0 complete: {len(pdf_paths)} document(s), {len(page_records)} page record(s) ready | output -> {OUTPUT_DIR.resolve()}")

## Step 1: Render pages at native scan resolution

**Why:** every sample page is one embedded JPEG scan (~200 DPI). Rendering at 300 DPI only enlarges it, and even rendering at "1×" resamples it and shifts it by 1 px (the image sits at a 1 pt offset).

**Method:**
- If a page is exactly **one image covering the whole page** (no rotation or flips), decode that image directly. This gives the exact scanned pixels, in BGR order ready for OpenCV.
- Otherwise render the page with pdfium at `fallback_dpi` and flag it.
- Optional enlargement (`CONFIG["render"]["upscale"]`, capped by the device profile). Off by default.
- **No filtering.** `images["original"]` is kept untouched for before/after comparison.

**Metrics:** extraction method, pixel size, orientation (from pixels), estimated DPI, coloured-ink share (`color_ink_pct`), paper tint (`paper_tint`), and whether the scan is pure greyscale.

The PDF reports 72 DPI, which is meaningless, so DPI is *estimated* from the short side assuming 8.5-inch-wide paper.

**Review flags:** `low_resolution`, `not_single_scan` (rendered instead of extracted), `page_rotation_set`, `partial_image_coverage`.

In [ ]:
# Step 1 (code: image_processing.render_page)
# pdfium is not thread-safe and extraction takes ~10 ms/page, so this step runs sequentially.
t_start = time.perf_counter()
for doc_path in dict.fromkeys(r["doc_path"] for r in page_records):
    doc = pdfium.PdfDocument(doc_path)
    for rec in (r for r in page_records if r["doc_path"] == doc_path):
        rec["images"]["original"] = render_page(
            doc[rec["page_index"]], rec, CONFIG["render"], CONFIG["review"]["render"], PROFILE["max_render_scale"]
        )
    doc.close()

# One DPI per document (median of all its pages, same function the module uses) unless overridden
if CONFIG["render"]["dpi_per_document"] and not CONFIG["render"]["dpi_override"]:
    for doc_path in dict.fromkeys(r["doc_path"] for r in page_records):
        doc = pdfium.PdfDocument(doc_path)
        doc_dpi = document_dpi(doc)
        doc.close()
        if doc_dpi:
            for r in (r for r in page_records if r["doc_path"] == doc_path):
                r["dpi"] = doc_dpi * r["metrics"][STEP1]["upscale"]
                r["metrics"][STEP1]["dpi_used"] = round(r["dpi"])

elapsed = time.perf_counter() - t_start
total_mb = sum(r["images"]["original"].nbytes for r in page_records) / 1024**2
print(f"Step 1 complete: {len(page_records)} pages in {elapsed:.2f}s | originals in memory: {total_mb:.0f} MB")

In [ ]:
# ---------- Step 1 review: per-page table + thumbnails ----------
print(f"{'Page':<18}{'Method':<16}{'Pixels (WxH)':<14}{'Orient':<11}{'DPI':>5}{'Scan':>8}{'Ink%':>7}{'Tint':>6}{'ms':>7}  Flags")
for rec in page_records:
    m = rec["metrics"][STEP1]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP1) or "-"
    w, h = m["native_px"]
    print(f"{page_label(rec):<18}{m['method']:<16}{f'{w}x{h}':<14}{m['orientation']:<11}{m['dpi_used']:>5}"
          f"{('grey' if m['greyscale_scan'] else 'colour'):>8}{m['color_ink_pct']:>7.2f}{m['paper_tint']:>6.0f}{m['time_ms']:>7}  {flags}")

step1_flagged = [r for r in page_records if any(f["step"] == STEP1 for f in r["flags"])]
print(f"\nFlagged for review in Step 1: {len(step1_flagged)} / {len(page_records)} pages")
for rec in step1_flagged:
    for f in rec["flags"]:
        if f["step"] == STEP1:
            print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")

for doc_name in dict.fromkeys(r["doc"] for r in page_records):
    show_page_grid([r for r in page_records if r["doc"] == doc_name], "original", f"{doc_name}: original (native resolution)")

## Step 2: Page quality assessment (measure only)

Nothing is changed in this step. It measures each page and decides **which later steps the page needs**, so every cleaning step runs only where it helps.

| Metric | How it is measured | Drives |
|---|---|---|
| `borders` | Per side, in the outer 20%: **dark strips** (≥ 3 lines where ≥ 35% of pixels are dark, e.g. a book spine, even when it sits a little inside the edge) and, in the outer 8%, **pale strips** (lines that are evenly off-paper in colour, e.g. a greenish scanner-bed edge). | Step 4 |
| `content_box` | The page minus those border bands. All metrics below are measured inside it. | all |
| `ink_pct` | Stroke pixels (morphological black-hat > 30: locally darker, thinner than 15 px), as % of the content area. | blank detection |
| `sharpness` | High-frequency energy on stroke edges ÷ stroke contrast. Real pages 5–18; synthetic Gaussian blur σ 1.5 ≈ 4–5. | Step 8 |
| `noise_sigma` | Immerkær noise estimate on flat (non-edge) pixels. The scanner clips the paper to white, so clean pages are ≈ 0–1. | Step 7 |
| `texture_pct` | Faint small dots not near any real stroke (toner grain, paper texture). | Step 7 |
| `isolated_specks_per_mp` | Tiny ink blobs (≤ 6 px) with no other ink within 8 px. Dotted fill-in lines don't count. | Step 7 |
| `faint_stroke_pct` | Faint, curvy, stroke-shaped marks not near real ink, after removing ruled/dotted lines. This is the bleed-through signal. | Step 9 |
| `illumination_spread` | 95th − 5th percentile of the estimated paper level (shadows, uneven lighting). | Step 6 |
| `paper_lab`, `paper_tint` | Median Lab colour of paper pixels; chroma distance from neutral grey. | Step 6 |
| `ink_level`, `ink_contrast` | Median grey of strokes; paper − ink. | Step 10 |
| `ink_colors` | Share of strokes that are blue / purple / red / black (by chroma + hue). | Step 6 (keep ink colour) |

**Review flags:** `blank`, `near_blank`, `blurry`, `noisy`, `many_specks`, `bleed_through_suspected`, `large_border`, `uneven_illumination`.

In [ ]:
# code: image_processing.step2_assess
run_parallel(step2_assess, page_records, "Step 2 complete")

In [ ]:
# ---------- Step 2 review: metrics table, what each page needs, flagged pages ----------
NEED_SHORT = {"border_cleanup": "B", "illumination": "I", "tint_removal": "T", "speck_removal": "S",
              "sharpen": "Sh", "bleed_suppression": "Bl", "contrast": "C"}


def needs_str(n: dict) -> str:
    if n["skip_page"]:
        return "SKIP (blank)"
    parts = [NEED_SHORT[k] for k in NEED_SHORT if n[k]]
    if n["denoise"] != "none":
        parts.append(f"D:{n['denoise']}")
    return " ".join(parts) or "-"


print(f"{'Page':<16}{'Ink%':>6}{'Sharp':>7}{'Noise':>7}{'Text%':>7}{'Speck':>7}{'Faint%':>8}{'Illum':>7}{'Tint':>6}"
      f"{'InkL':>6}{'Blue%':>7}  {'Borders (px)':<26}{'Needs':<22}Flags")
for rec in page_records:
    m = rec["metrics"][STEP2]
    b = " ".join(f"{s[0].upper()}{v['px']}{ {'dark': 'd', 'pale': 'p', 'dark+pale': 'dp'}[v['kind']] }"
                 for s, v in m["borders"].items() if v["px"]) or "-"
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP2) or "-"
    print(f"{page_label(rec):<16}{m['ink_pct']:>6.1f}{m['sharpness']:>7.1f}{m['noise_sigma']:>7.2f}{m['texture_pct']:>7.2f}"
          f"{m['isolated_specks_per_mp']:>7}{m['faint_stroke_pct']:>8.2f}{m['illumination_spread']:>7.0f}{m['paper_tint']:>6.1f}"
          f"{m['ink_level']:>6}{m['ink_colors']['blue']:>7.1f}  {b:<26}{needs_str(rec['needs']):<22}{flags}")
print("\nBorders: L/R/T/B + px + kind (d = dark strip, p = pale strip, dp = both). "
      "Needs: B border, I illumination, T tint, D denoise, S specks, Sh sharpen, Bl bleed, C contrast.")

# How many pages need each later step
print("\nPages needing each step:")
for key in ["border_cleanup", "illumination", "tint_removal", "speck_removal", "sharpen", "bleed_suppression", "contrast"]:
    print(f"  {key:<18} {sum(bool(r['needs'][key]) for r in page_records):>3} / {len(page_records)}")
for level in ("mild", "strong"):
    print(f"  denoise={level:<10} {sum(r['needs']['denoise'] == level for r in page_records):>3} / {len(page_records)}")

print("\nObserved range in this run vs CONFIG expected_ranges (update ranges only after reviewing outliers):")
for key, (lo, hi) in CONFIG["assess"]["expected_ranges"].items():
    vals = [r["metrics"][STEP2][key] for r in page_records]
    print(f"  {key:<24} observed [{min(vals):>7}, {max(vals):>7}]   expected [{lo}, {hi}]")

step2_flagged = [r for r in page_records if any(f["step"] == STEP2 for f in r["flags"])]
print(f"\nFlagged for review in Step 2: {len(step2_flagged)} / {len(page_records)} pages")
for rec in step2_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP2):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")

In [ ]:
# ---------- Step 2 visual check: detected border bands (red) and content box (green) on flagged pages ----------
def step2_overlay(rec: dict, max_h: int = 700) -> np.ndarray:
    img = rec["images"]["original"]
    h, w = img.shape[:2]
    s = max_h / h
    vis = cv2.resize(img, (int(w * s), max_h), interpolation=cv2.INTER_AREA)
    shade = vis.copy()
    x0, y0, x1, y1 = (int(v * s) for v in rec["metrics"][STEP2]["content_box"])
    mask = np.ones(vis.shape[:2], bool)
    mask[y0:y1, x0:x1] = False
    shade[mask] = (0, 0, 255)
    vis = cv2.addWeighted(shade, 0.35, vis, 0.65, 0)
    cv2.rectangle(vis, (x0, y0), (x1 - 1, y1 - 1), (0, 160, 0), 2)
    return vis


def show_overlays(records: list[dict], title: str, cols: int = 6) -> None:
    if not records:
        print(f"{title}: none")
        return
    rows = (len(records) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 3.8), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, rec in zip(axes.flat, records):
        ax.imshow(cv2.cvtColor(step2_overlay(rec), cv2.COLOR_BGR2RGB))
        codes = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP2)
        ax.set_title(f"{page_label(rec)}\n{codes}", fontsize=8, color="red")
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


show_overlays(step2_flagged, "Step 2 flagged pages: red = detected border band, green = content box")

## Step 3: Orientation by content (0° / 90° / 180° / 270°)

**Rule:** turn a page only when its **text** is sideways or upside down. A landscape page whose text reads normally stays landscape. Only exact 90° turns are used (`cv2.rotate`, lossless); small tilts are Step 5. OpenCV only, no OCR.

**How:**
1. **Text mask** of the content box (Step 2) at 150 DPI: strokes (black-hat), with ruled and dotted lines removed.
2. **Axis (horizontal or vertical text):** close the mask along x and, separately, along y. Horizontal text turns into long flat blobs only when smeared along x.
   - `axis_score` = share of ink in long horizontal blobs − share in long vertical blobs.
   - On the samples, upright pages score ≥ +0.02 and the same pages turned 90° score ≤ −0.02 (all 142 cases correct).
3. **Up/down:** find horizontal bars (Devanagari heading bar शिरोरेखा, T-bars). Upright text has more ink hanging *below* a bar than above it.
   - `updown_score` = vote(page) − vote(page turned 180°).
   - This is reliable for Hindi text but weak for English (the English medical pages score −0.13 to −0.40). So the step is **conservative**: it rotates 180° only at ≤ −0.5 and flags at ≤ −0.15.

**Decisions:**
- Clearly vertical text → turn 90° the way the up/down vote prefers, and flag `rotated`.
- Horizontal text with a strong upside-down vote → turn 180° and flag `rotated`.
- Weak evidence → **no turn**, flag `orientation_uncertain` or `possibly_upside_down` for review.

**Output:** `images["oriented"]`. The Step 2 `content_box` and `borders` are carried over into the new orientation (`metrics["step3_orient"]`).

In [ ]:
# code: image_processing.step3_orient
run_parallel(step3_orient, page_records, "Step 3 complete")

In [ ]:
# ---------- Step 3 review: decisions, flagged pages, optional self-test ----------
print(f"{'Page':<16}{'Turn':>6}  {'Decision':<26}{'Axis':>7}{'Up/down':>9}  {'Before -> after':<22}Flags")
for rec in page_records:
    m = rec["metrics"][STEP3]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP3) or "-"
    fmt = lambda v: "-" if v is None else f"{v:+.3f}"
    print(f"{page_label(rec):<16}{m['rotation']:>5}°  {m['decision']:<26}{fmt(m['axis_score']):>7}{fmt(m['updown_score']):>9}  "
          f"{m['orientation_before'] + ' -> ' + m['orientation_after']:<22}{flags}")

turned = [r for r in page_records if r["metrics"][STEP3]["rotation"]]
step3_flagged = [r for r in page_records if any(f["step"] == STEP3 for f in r["flags"])]
print(f"\nTurned: {len(turned)} | flagged for review: {len(step3_flagged)} / {len(page_records)}")
for rec in step3_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP3):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")

if step3_flagged:
    show_page_grid(step3_flagged, "oriented", "Step 3 flagged pages (after orientation)", cols=6)

# Self-test: turn real pages by a known angle and check the detector undoes it (slow: 3 extra passes per page)
RUN_ORIENTATION_SELF_TEST = False
if RUN_ORIENTATION_SELF_TEST:
    cfg = CONFIG["orient"]
    results = {"correct": 0, "not_turned": 0, "wrong_turn": 0}
    for rec in [r for r in page_records if not r["needs"]["skip_page"]]:
        base = orientation_text_mask(rec["images"]["original"], rec["metrics"][STEP2]["content_box"],
                                     rec["dpi"], CONFIG["assess"], cfg["work_dpi"])
        for applied in (90, 180, 270):
            got = detect_orientation(rotate(base, applied), cfg["work_dpi"], cfg)["rotation"]
            key = "correct" if (got + applied) % 360 == 0 else ("not_turned" if got == 0 else "wrong_turn")
            results[key] += 1
    print("Orientation self-test on turned copies:", results)

## Step 4: Border & scanner-artifact cleanup

**Why:** black book-spine and scanner shadows, slivers of the neighbouring page, and pale scanner-bed strips throw off the thresholds, deskew and background estimate in later steps.

**Input:** `images["oriented"]` and the border bands carried through Step 3. Only pages with `needs["border_cleanup"]` are changed.

**Method:**
1. **Fill, don't crop:** pale bands (scanner bed, off-page area) are replaced with the page's **paper colour** (Step 2 `paper_lab`), so the page keeps its size and Step 6 later whitens it together with the rest of the paper.
   **Dark bands (spine shadow) are flattened, not painted:** page text is often printed *on* the shadow (row numbers, the start of field labels). The local background (grey closing wider than any stroke) is divided out, so the shadow turns to paper while the text stays. Only background too dark to hold any text (below paper ÷ `shadow_max_gain`) is filled. The area beyond the spine is flattened too, **never erased**: on these samples it is sometimes a neighbour-page sliver and sometimes this page's own lines continuing across the spine, which pixels alone cannot tell apart. If it holds ink, the page is flagged `content_beyond_spine`.
2. **Protect text at the edge:** ink strokes (black-hat) inside the content box within `protect_mm` of a band are followed into the band. Any stroke that crosses the boundary (a descender, the last line on a short sheet) is restored from the original. Neighbour-page text outside a spine doesn't touch the content box, so it is still removed.
3. **Feather:** the fill blends in over `feather_mm` so later filters don't see a hard step.
4. **Verify:** Step 2's border detector runs again on the result. Any border still found is flagged.

**Review flags:** `text_at_border` (strokes crossed into a band and were kept), `content_beyond_spine`, `large_fill` (> 20% of the page filled), `border_remaining`.

**Safety (review): nothing inside a dark band is removed.** Both sides of the spine can hold text: this page's row numbers and field labels (inside the shadow) and the neighbour page's sliver (beyond it).
- Recoverable shadow → **flattened** (text stays), which is the reviewed default. An optional **monotone tone lift** inside the band (`shadow_lift: (0.45, 0.80)`, off by default) keeps lightness below 45% of paper (text) unchanged, turns lightness above 80% (shadow grain) into paper, and blends smoothly in between. Because the curve never reorders tones, no stroke can disappear; (0.40, 0.70) was judged too strong. *Rejected:* binary "text or not" masks (local-median background, hysteresis on the flattened band). They erased "9" of "9654", "(i)–(iv)", "Type of" and "Place of" on FIR-22-2010 p1.
- Too dark to flatten (darker than paper ÷ 6) → **left as the original pixels** (`fill_unrecoverable_dark: False`). Painting it with paper is cosmetic only and could hide white-on-black content, so it is off.
- Bright glyph-shaped marks on a solid dark background (≥ 60% black within 4 mm) are reported as `inverted_content_in_border` for review.
- Only **pale** bands (scanner bed) are ever painted, and pen strokes crossing into them are restored.

In [ ]:
# code: image_processing.step4_borders
run_parallel(step4_borders, page_records, "Step 4 complete")

In [ ]:
# ---------- Step 4 review: what was filled, flags, before/after ----------
changed = [r for r in page_records if r["metrics"][STEP4]["applied"]]
print(f"{'Page':<16}{'Filled %':>9}{'Flat %':>8}{'Kept mm²':>10}  {'Sides (px)':<36}{'Remaining':<16}Flags")
for rec in changed:
    m = rec["metrics"][STEP4]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP4) or "-"
    sides = " ".join(f"{s[0].upper()}{v}" for s, v in m["sides"].items())
    rem = " ".join(f"{s[0].upper()}{v}" for s, v in m["remaining_after"].items()) or "-"
    print(f"{page_label(rec):<16}{m['filled_pct']:>9.1f}{m['shadow_flattened_pct']:>8.1f}{m['restored_mm2']:>10.1f}  "
          f"{sides:<36}{rem:<16}{flags}")
print(f"\nCleaned {len(changed)} / {len(page_records)} pages")

step4_flagged = [r for r in page_records if any(f["step"] == STEP4 for f in r["flags"])]
print(f"Flagged for review in Step 4: {len(step4_flagged)}")
for rec in step4_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP4):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")


def show_before_after(records: list[dict], before: str, after: str, title: str, step: str,
                      max_pages: int = 12, h: int = 520) -> None:
    """Side-by-side before|after thumbnails; titles show that step's flags."""
    records = records[:max_pages]
    if not records:
        return
    cols = 3
    rows = (len(records) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5.2, rows * 4.2), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, rec in zip(axes.flat, records):
        pair = []
        for key in (before, after):
            im = rec["images"][key]
            im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR) if im.ndim == 2 else im      # B&W pages
            s = h / im.shape[0]
            pair.append(cv2.resize(im, (int(im.shape[1] * s), h), interpolation=cv2.INTER_AREA))
        sep = np.full((h, 8, 3), 255, np.uint8)
        ax.imshow(cv2.cvtColor(np.hstack([pair[0], sep, pair[1]]), cv2.COLOR_BGR2RGB))
        codes = ", ".join(f["code"] for f in rec["flags"] if f["step"] == step)
        ax.set_title(f"{page_label(rec)}  (before | after)" + (f"\n⚑ {codes}" if codes else ""), fontsize=9,
                     color="red" if codes else "black")
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


# Largest changes first, flagged pages always included
biggest = sorted(changed, key=lambda r: -r["metrics"][STEP4]["filled_pct"])
show_list = step4_flagged + [r for r in biggest if r not in step4_flagged]
show_before_after(show_list, "oriented", "borders_clean", "Step 4: border cleanup (flagged + largest fills)", STEP4)

## Step 5: Deskew (small angle, ±10°)

**Why:** most scans are tilted by a fraction of a degree to a few degrees. Straight text lines help reading, later steps and any future OCR.

**Input:** `images["borders_clean"]` and the content box (Step 3). Measured on a 150 DPI copy of the content box only, so borders don't take part.

**Two independent estimates:**
1. **Projection (text):** the stroke mask with ruled/dotted lines removed is rotated through ±10°, coarse then fine (0.05°). The angle where text rows are sharpest (largest sum of squared row-to-row profile changes) wins. Works on handwritten pages.
2. **Printed lines (forms):** Hough segments ≥ 30 mm long (form rules, dotted fill-in lines, gaps ≤ 1.5 mm bridged), within ±10° of horizontal. Their angles are combined as a length-weighted median.

**Decision (thresholds measured with the self-test on tilted sample pages):**
- **Strong lines** (≥ 10 segments, ≥ 1 m total) → line angle. Median error 0.09°, 95% within 0.5°. This also works on handwriting, because Hough follows the long Devanagari heading bars.
- Otherwise **strong projection** (text rows ≥ 1.1× sharper at the best angle than at 0°) → projection angle. If there are some (weak) lines that differ from it by more than 1°, the page is treated as uncertain instead.
- Otherwise both weak but agreeing within 0.5° → line angle.
- Otherwise → **not rotated**, flagged `deskew_uncertain` if either estimate suggests ≥ 0.5°.
- Under 0.1° → leave the page alone.
- If the writing itself is slanted (lines rising 5–6° on a straight sheet), the step levels the **text**, because that is what matters for reading and extraction, and flags it `large_skew`.

**Rotation:** around the centre with an **expanded canvas** (no corner is clipped). New area gets the paper colour. Cubic interpolation, on the device's warp backend (OpenCL on the Mac, CUDA on GB10 when available). Portrait/landscape never changes.

**Review flags:** `deskew_disagree` (both estimates strong but different), `large_skew` (> 5°), `deskew_uncertain`.

In [ ]:
# code: image_processing.step5_deskew
run_parallel(step5_deskew, page_records, "Step 5 complete")

In [ ]:
# ---------- Step 5 review: angles, flags, before/after, optional self-test ----------
fmt = lambda v: "-" if v is None else f"{v:+.2f}"
print(f"{'Page':<16}{'Applied':>9}  {'Source':<11}{'Lines°':>8}{'#':>4}{'Text°':>8}{'Peak':>7}  {'Orient':<10}Flags")
for rec in page_records:
    m = rec["metrics"][STEP5]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP5) or "-"
    applied = f"{m['angle']:+.2f}°" if m["rotated"] else "-"
    print(f"{page_label(rec):<16}{applied:>9}  {m['source']:<11}{fmt(m['line_angle']):>8}{m['n_lines']:>4}"
          f"{fmt(m['projection_angle']):>8}{('-' if m['projection_peak'] is None else m['projection_peak']):>7}  "
          f"{m['orientation']:<10}{flags}")

rotated5 = [r for r in page_records if r["metrics"][STEP5]["rotated"]]
step5_flagged = [r for r in page_records if any(f["step"] == STEP5 for f in r["flags"])]
angles = [abs(r["metrics"][STEP5]["angle"]) for r in rotated5]
print(f"\nRotated {len(rotated5)} / {len(page_records)} pages"
      + (f" | |angle| median {np.median(angles):.2f}°, max {max(angles):.2f}°" if angles else ""))
print(f"Flagged for review in Step 5: {len(step5_flagged)}")
for rec in step5_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP5):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")

biggest5 = sorted(rotated5, key=lambda r: -abs(r["metrics"][STEP5]["angle"]))
show_before_after(step5_flagged + [r for r in biggest5 if r not in step5_flagged],
                  "borders_clean", "deskewed", "Step 5: deskew (flagged + largest angles)", STEP5)

# Self-test: tilt real pages by known angles and check the estimate (error = estimate + applied tilt)
RUN_DESKEW_SELF_TEST = False
if RUN_DESKEW_SELF_TEST:
    cfg = CONFIG["deskew"]
    errs = {"lines": [], "projection": []}
    for rec in [r for r in page_records if not r["needs"]["skip_page"]]:
        base = rec["images"]["deskewed"]
        for tilt in (-4.0, -1.5, -0.6, 0.6, 1.5, 4.0):
            tilted, M = rotate_expand(base, tilt, (255.0, 255.0, 255.0))
            box = transform_box(rec["metrics"][STEP5]["content_box"], M, (tilted.shape[1], tilted.shape[0]))
            est = estimate_skew(tilted, box, rec["dpi"], cfg)
            key = "lines" if est["source"] == "lines" else ("projection" if est["source"] == "projection" else "other")
            errs.setdefault(key, []).append(abs(est["angle"] + tilt))
    for src, e in errs.items():
        if e:
            e = np.array(e)
            print(f"Self-test [{src}]: {len(e)} cases | median error {np.median(e):.2f}° | 95th pct {np.percentile(e, 95):.2f}° "
                  f"| > 0.5°: {(e > 0.5).sum()}")

## Step 6: Illumination & paper-colour normalization

**Why:** shadows, yellowed paper and pink/lavender sheets lower contrast and make every page a different background. Later steps and any reader work better on white paper.

**Method (local white balance):**
1. **Paper colour everywhere:** on a 50 DPI copy, each colour channel is closed with a 12 mm window. This removes strokes, dense handwriting and black banner bars, which are all thinner than 12 mm. Median and Gaussian smoothing follow, then the estimate is resized back to full size.
2. **Divide it out per channel:** `out = image × 255 / paper`. Paper becomes white whatever its tint or shading. Ink colours are kept *relative to the paper*, so blue ink stays blue and purple stamps stay purple.
3. **Safety cap:** gain limited to 0.95–1.5×, so dark areas wider than the window (thumbprints, big stamps) are not bleached. If more than 1% of the page hits the cap, it is reported.
4. **Soft white point:** luminance 235–255 is eased to pure white (no hard threshold). Luminance, not the brightest channel: blue ink has a bright blue channel but is dark.

Applied to every non-blank page (`apply_to_all_pages`), so all pages share one white. Pages that were already white change by almost nothing.

**Verified per page (before → after):** paper lightness and tint, paper-level spread, stroke-pixel count (fading), and the blue/purple/red share of strokes. A share that *drops* is colour loss; a rise is expected, because ink on yellow or pink paper regains its true colour.

**Review flags:** `ink_loss`, `colour_loss`, `tint_remaining`, `illumination_remaining`, `gain_capped`.

**Safety (review):**
- Where the paper estimate is darker than 55% of the page's paper, it is an **object** (photo, thumbprint, seal, black banner), not paper. The estimate there is filled in (inpainted) from the surrounding paper, so the object keeps its tones; this is reported as `dark_object_kept`.
- The white point starts at luminance 245 (was 235), so pencil marks and very light stamps are not faded.

In [ ]:
# code: image_processing.step6_illumination
run_parallel(step6_illumination, page_records, "Step 6 complete")

In [ ]:
# ---------- Step 6 review: before -> after per page, flags, before/after images ----------
print(f"{'Page':<16}{'Need':>5}{'Paper L':>15}{'Tint':>13}{'Spread':>13}{'Ink Δ%':>8}{'Col -':>7}{'Cap %':>7}  Flags")
for rec in page_records:
    m = rec["metrics"][STEP6]
    if not m["applied"]:
        print(f"{page_label(rec):<16}{'-':>5}  not applied")
        continue
    b, a = m["before"], m["after"]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP6) or "-"
    print(f"{page_label(rec):<16}{('yes' if m['needed'] else '-'):>5}{b['paper_L']:>7}->{a['paper_L']:<6}"
          f"{b['paper_tint']:>6}->{a['paper_tint']:<5}{b['illumination_spread']:>6}->{a['illumination_spread']:<5}"
          f"{m['ink_change_pct']:>8}{m['colour_loss_pts']:>7}{m['gain_capped_pct']:>7}  {flags}")

applied6 = [r for r in page_records if r["metrics"][STEP6]["applied"]]
if applied6:
    for key in ("paper_L", "paper_tint", "illumination_spread"):
        b = [r["metrics"][STEP6]["before"][key] for r in applied6]
        a = [r["metrics"][STEP6]["after"][key] for r in applied6]
        print(f"{key:<20} median {np.median(b):.1f} -> {np.median(a):.1f} | worst {max(b):.1f} -> {max(a):.1f}")
step6_flagged = [r for r in page_records if any(f["step"] == STEP6 for f in r["flags"])]
print(f"Flagged for review in Step 6: {len(step6_flagged)}")
for rec in step6_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP6):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")

# Flagged pages first, then the pages that changed most (tint / lighting)
changed6 = sorted(applied6, key=lambda r: -(r["metrics"][STEP6]["before"]["paper_tint"]
                                           + r["metrics"][STEP6]["before"]["illumination_spread"]))
show_before_after(step6_flagged + [r for r in changed6 if r not in step6_flagged], "deskewed", "illum_norm",
                  "Step 6: illumination & paper colour (flagged + biggest changes)", STEP6)

## Step 7: Noise removal (adaptive)

**Why:** photocopy grain, grey toner texture, grain amplified when Step 4 flattened spine shadows, colour speckle from JPEG, and isolated specks. Thin handwriting must survive.

**Detect on the current image:** Step 2's `assess_page` runs again on `images["illum_norm"]`, because the Step 2 numbers predate the Step 4 flattening. Each fix runs only where its measurement says so:

| Found | Fix |
|---|---|
| `noise_sigma` ≥ 1.0 (mild) / ≥ 2.0 (strong), or `texture_pct` ≥ 2.0 | Non-local means on **lightness only** (Lab L), h = 6 / 12, patch 0.9 mm, search 2.7 mm. Device backend: OpenCL / CUDA / CPU. |
| any denoise on a colour page | 0.6 mm median on Lab a/b (colour speckle; ink colour spans more pixels) |
| `texture_pct` ≥ 0.5 | Faint **dot-like** marks away from real strokes (the blobs Step 2 counts as texture) → paper. Ruled/dotted lines and faint **stroke-shaped** marks are kept (Step 9 handles bleed-through). |
| `isolated_specks_per_mp` ≥ 100 | Blobs ≤ 0.1 mm² with no other ink within ~2 mm → paper. Dotted lines, full stops and matras all have neighbours, so they stay. |

**Verify:** `assess_page` runs again after the fix: noise, texture, specks and ink before → after.

**Review flags:** `ink_loss` (pixels of real strokes, connected ink ≥ 0.5 mm², dropped > 8%; grain is excluded because removing it is the point), `noise_remaining`, `specks_remaining`.

**Known limit:** coarse grain left in spine shadows that Step 4 flattened (FIR-22-2010 p1/p3) is not removed. Non-local means leaves it, and removing small clusters inside the flattened zone was tried and reverted: the grain joins into larger clusters and survives, while the dots of dotted fill-in lines next to the spine were erased. Those pages stay flagged `noise_remaining`. The zoom view shows a full-resolution crop of the area that changed most on each page.

**Safety (review):**
- Texture and speck clearing only ever touches **neutral grey** marks; texture also only **faint** ones (black-hat ≤ 30). Stamps, seals and coloured ink dots are never erased.
- The colour median is applied to **paper pixels only**, so thin coloured strokes keep their colour.

In [ ]:
# code: image_processing.step7_denoise
run_parallel(step7_denoise, page_records, "Step 7 complete")

In [ ]:
# ---------- Step 7 review: plan + before -> after, flags, zoomed crops where the change was largest ----------
print(f"{'Page':<16}{'NLM':<8}{'Chr':>4}{'Tex':>4}{'Spk':>4}{'Noise':>14}{'Texture %':>14}{'Specks/MP':>13}{'All ink %':>14}{'Strokes Δ%':>11}  Flags")
for rec in page_records:
    m = rec["metrics"][STEP7]
    if not m.get("applied"):
        continue
    p, b, a = m["plan"], m["before"], m["after"]
    yn = lambda v: "y" if v else "-"
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP7) or "-"
    print(f"{page_label(rec):<16}{p['nlm']:<8}{yn(p['chroma']):>4}{yn(p['texture']):>4}{yn(p['specks']):>4}"
          f"{b['noise_sigma']:>7}->{a['noise_sigma']:<6}{b['texture_pct']:>7}->{a['texture_pct']:<6}"
          f"{b['isolated_specks_per_mp']:>6}->{a['isolated_specks_per_mp']:<6}{b['ink_pct']:>7}->{a['ink_pct']:<6}"
          f"{m['ink_change_pct']:>11}  {flags}")
applied7 = [r for r in page_records if r["metrics"][STEP7].get("applied")]
print(f"\nDenoised {len(applied7)} / {len(page_records)} pages "
      f"(strong {sum(r['metrics'][STEP7]['plan']['nlm'] == 'strong' for r in applied7)}, "
      f"mild {sum(r['metrics'][STEP7]['plan']['nlm'] == 'mild' for r in applied7)}, "
      f"texture {sum(r['metrics'][STEP7]['plan']['texture'] for r in applied7)}, "
      f"specks {sum(r['metrics'][STEP7]['plan']['specks'] for r in applied7)})")
step7_flagged = [r for r in page_records if any(f["step"] == STEP7 for f in r["flags"])]
print(f"Flagged for review in Step 7: {len(step7_flagged)}")
for rec in step7_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP7):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")


def show_zoom_pairs(records: list[dict], before: str, after: str, title: str, step: str,
                    crop_mm: float = 45, max_pages: int = 8) -> None:
    """Full-resolution before|after crops centred where the two images differ most (noise is invisible in thumbnails)."""
    records = records[:max_pages]
    if not records:
        return
    fig, axes = plt.subplots(len(records), 1, figsize=(14, 4.2 * len(records)), squeeze=False)
    for ax, rec in zip(axes[:, 0], records):
        a, b = rec["images"][before], rec["images"][after]
        if a.shape != b.shape:
            ax.axis("off"); continue
        diff = cv2.absdiff(cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)).astype(np.float32)
        c = mm_to_px(crop_mm, rec["dpi"])
        score = cv2.boxFilter(diff, -1, (c, c))
        y, x = np.unravel_index(int(np.argmax(score)), score.shape)
        y0 = int(np.clip(y - c // 2, 0, max(0, a.shape[0] - c))); x0 = int(np.clip(x - c, 0, max(0, a.shape[1] - 2 * c)))
        crop = lambda im: im[y0:y0 + c, x0:x0 + 2 * c]
        ax.imshow(cv2.cvtColor(np.hstack([crop(a), np.full((crop(a).shape[0], 8, 3), 0, np.uint8), crop(b)]), cv2.COLOR_BGR2RGB))
        codes = ", ".join(f["code"] for f in rec["flags"] if f["step"] == step)
        ax.set_title(f"{page_label(rec)}  (before | after, 1:1 crop)" + (f"  ⚑ {codes}" if codes else ""), fontsize=10,
                     color="red" if codes else "black")
        ax.axis("off")
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


most7 = sorted(applied7, key=lambda r: -(r["metrics"][STEP7]["before"]["noise_sigma"] + r["metrics"][STEP7]["before"]["texture_pct"]))
show_zoom_pairs(step7_flagged + [r for r in most7 if r not in step7_flagged], "illum_norm", "denoised",
                "Step 7: noise removal, zoomed where the change is largest", STEP7)

## Step 8: Deblur / sharpen (adaptive)

**Why:** some pages are soft: out-of-focus scans, or ink that has spread into the paper. Sharpening helps reading, but it must not sharpen noise or draw halos around strokes, and it must not touch pages that are already sharp.

**Detect on the current image:** `assess_page` again on `images["denoised"]` (sharpness = high-frequency energy on stroke edges ÷ stroke contrast, measured at 200 DPI). Sharpen only if sharpness < `sharpen_below` (7.0).

**Method (unsharp mask on lightness only):**
- `detail = L − Gaussian(L, 0.2 mm)`.
- Detail smaller than 4 grey levels is not boosted (paper texture, residual noise).
- **Strength from the deficit:** `amount = 3 × (9 − sharpness) / 9`, clipped to 0.3–1.5, so a slightly soft page gets a light touch.
- **Halo clamp:** each pixel stays within its 3×3 neighbourhood's min/max ± 8 grey levels.
- Lab a/b (ink colour) untouched.

**Verify:** sharpness, noise and real-stroke pixels before → after.

**Review flags:** `still_blurry` (< 5.0 after), `noise_amplified` (σ +0.5), `ink_change` (real strokes ±10%).

**Self-test (optional):** blur sharp pages by a known amount, sharpen, and compare.

In [ ]:
# code: image_processing.step8_sharpen
run_parallel(step8_sharpen, page_records, "Step 8 complete")

In [ ]:
# ---------- Step 8 review: sharpness of every page, what was sharpened, zoom, optional self-test ----------
sharp_now = sorted(((r["metrics"][STEP8]["sharpness"] if not r["metrics"][STEP8].get("applied")
                     else r["metrics"][STEP8]["sharpness"][0]), page_label(r))
                   for r in page_records if "sharpness" in r["metrics"][STEP8])
print("Sharpness before Step 8 (lowest first):", ", ".join(f"{lbl} {s}" for s, lbl in sharp_now[:10]), "...")
applied8 = [r for r in page_records if r["metrics"][STEP8].get("applied")]
print(f"\n{'Page':<16}{'Amount':>7}{'Sharpness':>16}{'Noise':>14}{'Strokes Δ%':>11}  Flags")
for rec in applied8:
    m = rec["metrics"][STEP8]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP8) or "-"
    print(f"{page_label(rec):<16}{m['amount']:>7}{m['sharpness'][0]:>8}->{m['sharpness'][1]:<6}"
          f"{m['noise_sigma'][0]:>7}->{m['noise_sigma'][1]:<6}{m['stroke_change_pct']:>11}  {flags}")
print(f"\nSharpened {len(applied8)} / {len(page_records)} pages")
step8_flagged = [r for r in page_records if any(f["step"] == STEP8 for f in r["flags"])]
print(f"Flagged for review in Step 8: {len(step8_flagged)}")
for rec in step8_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP8):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")
show_zoom_pairs(step8_flagged + [r for r in applied8 if r not in step8_flagged], "denoised", "sharpened",
                "Step 8: sharpening, zoomed where the change is largest", STEP8, crop_mm=35)

# Self-test: blur sharp pages by a known Gaussian, sharpen, compare sharpness (original -> blurred -> restored)
RUN_SHARPEN_SELF_TEST = False
if RUN_SHARPEN_SELF_TEST:
    cfg = CONFIG["sharpen"]
    rows = []
    for rec in [r for r in page_records if not r["needs"]["skip_page"]][::4]:
        base = rec["images"]["sharpened"]
        s0 = assess_page(base, CONFIG["assess"], rec["dpi"])["sharpness"]
        for sigma_mm in (0.1, 0.2):
            blurred = cv2.GaussianBlur(base, (0, 0), sigma_mm * rec["dpi"] / 25.4)
            sb = assess_page(blurred, CONFIG["assess"], rec["dpi"])["sharpness"]
            restored = sharpen_luminance(blurred, rec["dpi"], sharpen_amount(sb, cfg), cfg) if sb < CONFIG["assess"]["needs"]["sharpen_below"] else blurred
            sr = assess_page(restored, CONFIG["assess"], rec["dpi"])["sharpness"]
            rows.append((page_label(rec), sigma_mm, s0, sb, sr))
    for lbl, sg, s0, sb, sr in rows:
        print(f"  {lbl:<16} blur {sg} mm: {s0} -> blurred {sb} -> restored {sr}")
    rec_frac = [(sr - sb) / max(0.01, s0 - sb) for _, _, s0, sb, sr in rows if s0 - sb > 0.5]
    if rec_frac:
        print(f"Self-test: median recovery of lost sharpness {np.median(rec_frac) * 100:.0f}%")

## Step 9: Bleed-through suppression

**Why:** writing from the back of the sheet shows through (FIR-2-2009 p8, p9, p5) and competes with the real text.

**What bleed-through looks like here:** the mirrored back-side writing is not "faint" by the stroke measure. It has clear edges, and it is the **same ink colour as the front writing, only lighter**. Front blue ink on p8 has a mean lightness of about 174; the show-through is about 214.

**Detect (every page):**
1. Ink components (black-hat strokes ≥ 0.3 mm²) are grouped by colour: blue / purple / red (hue bands from `CONFIG["assess"]["ink_hues"]`).
2. In each group, component lightness is split with Otsu. It counts as bleed-through only if the split is clean: separability η ≥ 0.8, gap ≥ 30 L, light share 25–85%, ≥ 100 components.
3. Black/grey ink is **not** tested: printed text and grey dotted lines always split, even on clean forms.

On the samples: bleed pages η 0.88–0.92 with a gap of 39–58. Normal blue pages η 0.47–0.69, with a gap of 12–16 or a light share ≤ 14%.

**Fix (per pixel, not per component):**
- Pixels of that colour that are **lighter than the split** are blended **85% toward paper**: lightened, not erased.
- Pixels within 0.25 mm of **dark ink of that colour** (front strokes and their soft edges) or of **black/grey ink** (print) are never touched.

*Rejected:* classifying whole connected components. Bleed touches front strokes and print, so they merged into one "light blue" component. That lightened a dark front stroke on FIR-2-2009 p8 and erased printed text on p9.

**Review flags:** **every page this step changes is flagged `bleed_suppressed`**. Also `ink_change` if the dark (front) strokes changed by more than 5%.

**Limits:**
- Show-through that overlaps the front writing is part of the same components and stays.
- Black-ink bleed-through is not handled.
- Step 2's faint-stroke trigger is reported for comparison only.

**Safety (review):** lightening is 60% (was 85%). Show-through fades, but if real lighter writing were ever misdetected it stays readable.

In [ ]:
# code: image_processing.step9_bleed
run_parallel(step9_bleed, page_records, "Step 9 complete")

In [ ]:
# ---------- Step 9 review: colour-group lightness split per page, what was lightened, zoom ----------
print(f"{'Page':<16}{'Step2?':>7}  Groups tested (n, eta, gap, light share)                     Applied  Flags")
for rec in page_records:
    m = rec["metrics"][STEP9]
    if "groups" not in m:
        continue
    g = "; ".join(f"{k}: n={v['n']}" + (f" eta={v['eta']} gap={v['gap']} share={v['light_share']}{' BLEED' if v['bleed'] else ''}"
                                        if "eta" in v else "") for k, v in m["groups"].items() if v["n"])
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP9) or "-"
    if m["applied"] or m["step2_suspected"] or any(v.get("eta", 0) >= 0.7 for v in m["groups"].values()):
        print(f"{page_label(rec):<16}{('yes' if m['step2_suspected'] else '-'):>7}  {g:<60}{('yes' if m['applied'] else '-'):>8}  {flags}")
applied9 = [r for r in page_records if r["metrics"][STEP9].get("applied")]
print(f"\nBleed-through lightened on {len(applied9)} / {len(page_records)} pages (all flagged for review). "
      "Pages with no colour group near the split are not listed.")
for rec in applied9:
    m = rec["metrics"][STEP9]
    print(f"  {page_label(rec)}: lightened {m['lightened_pct']}% of page, front strokes {m['front_change_pct']:+}%")
show_before_after(applied9, "sharpened", "bleed_clean", "Step 9: bleed-through (before | after)", STEP9)
show_zoom_pairs(applied9, "sharpened", "bleed_clean", "Step 9: bleed-through, zoomed where the change is largest",
                STEP9, crop_mm=60)

## Step 10: Contrast enhancement (faint ink)

**Why:** faint ballpoint, carbon copies and washed-out blue ink stay hard to read even on white paper.

**Detect on the current image:** `assess_page` on `images["bleed_clean"]`. Act only if the median grey of the strokes (`ink_level`) is ≥ 180.

**Method (adaptive tone curve on Lab L only):**
- Below a knee at L = 235: `L' = knee × (L / knee)^γ`. At and above the knee: unchanged.
- γ is chosen so the page's median ink reaches 140 (`target_ink_level`), capped at 2.5.
- Paper (255 after Step 6) stays exactly white. Near-white texture above the knee is not darkened. The curve is monotone (no reordering, no new edges). Colour (a/b) is untouched, so blue stays blue.
- *Not CLAHE:* the paper is already uniform after Step 6, and local histogram equalisation would bring back paper texture.

**Verify:** ink level, noise and texture before → after.

**Review flags:** `still_faint` (median ink still ≥ 170), `noise_amplified` (noise σ +0.5 or texture +0.5%).

In [ ]:
# code: image_processing.step10_contrast
run_parallel(step10_contrast, page_records, "Step 10 complete")

In [ ]:
# ---------- Step 10 review: ink level of every page, what was enhanced, zoom ----------
lv = sorted(((r["metrics"][STEP10]["ink_level"] if not r["metrics"][STEP10].get("applied")
              else r["metrics"][STEP10]["ink_level"][0]), page_label(r))
            for r in page_records if "ink_level" in r["metrics"][STEP10])
print("Median ink grey before Step 10 (faintest first):", ", ".join(f"{l} {v}" for v, l in lv[::-1][:12]), "...")
applied10 = [r for r in page_records if r["metrics"][STEP10].get("applied")]
print(f"\n{'Page':<16}{'Gamma':>6}{'Ink grey':>14}{'Noise':>14}{'Texture %':>14}  Flags")
for rec in applied10:
    m = rec["metrics"][STEP10]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP10) or "-"
    print(f"{page_label(rec):<16}{m['gamma']:>6}{m['ink_level'][0]:>7}->{m['ink_level'][1]:<5}"
          f"{m['noise_sigma'][0]:>7}->{m['noise_sigma'][1]:<6}{m['texture_pct'][0]:>7}->{m['texture_pct'][1]:<6}  {flags}")
print(f"\nEnhanced {len(applied10)} / {len(page_records)} pages")
step10_flagged = [r for r in page_records if any(f["step"] == STEP10 for f in r["flags"])]
print(f"Flagged for review in Step 10: {len(step10_flagged)}")
for rec in step10_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP10):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")
show_before_after(applied10, "bleed_clean", "contrast", "Step 10: faint-ink contrast (before | after)", STEP10)
show_zoom_pairs(applied10, "bleed_clean", "contrast", "Step 10: zoomed where the change is largest", STEP10, crop_mm=45)

## Step 11: Final image and saved files

`finalize_page` sets the final colour image. `save_page` writes, flat in `OUTPUT_DIR` (no per-PDF folder):

| File | What |
|---|---|
| `{pdf}_{page}_original.jpg` | Exact embedded scan, byte for byte (`.png` if the page had to be rendered) |
| `{pdf}_{page}_final.png` | Cleaned colour page, lossless: the image to send to the model |
| `{pdf}_{page}_applied.json` | Every step in pipeline order: applied yes/no, why, parameters used, result, flags, needs_review |

The page number is zero-padded to at least 3 digits.

A black & white version (Sauvola) is **off** (`CONFIG["output"]["bw"]["enabled"]`): colour keeps faint ink, stamps and ink colour for the model. When enabled, it is saved as `{pdf}_{page}_final_bw.png`, and its checks `bw_ink_loss` / `bw_noise` are added.

In [ ]:
# code: image_processing.finalize_page / save_page
run_parallel(finalize_page, page_records, "Step 11 final images ready")
for rec in page_records:
    save_page(rec, OUTPUT_DIR)
print(f"Saved {{pdf}}_{{page}}_original / _final.png / _applied.json for {len(page_records)} pages in {OUTPUT_DIR.resolve()}")

In [ ]:
# ---------- Step 11 review: pages needing review, files written, original | final ----------
for doc_name in dict.fromkeys(r["doc"] for r in page_records):
    recs = [r for r in page_records if r["doc"] == doc_name]
    review = [r["page_no"] for r in recs if r["flags"]]
    print(f"{doc_name}: {len(recs)} pages, {len(review)} need review {review}")
size_mb = sum((OUTPUT_DIR / f).stat().st_size for r in page_records for f in r["files"].values()) / 1024**2
print(f"Files written: {sum(len(r['files']) for r in page_records)} ({size_mb:.0f} MB) in {OUTPUT_DIR.resolve()}")
print("Example:", page_records[0]["files"])
flagged = [r for r in page_records if r["flags"]]
show_before_after((flagged + [r for r in page_records if r not in flagged])[:9], "original", "final",
                  "Original | final (pages needing review first)", STEP11, max_pages=9)

## Using the module (production)

`DocumentPreprocessor` runs Steps 1–11 page by page, a few pages at a time, freeing intermediate images. Its output is pixel-identical to the step-by-step cells above.

```python
pre = DocumentPreprocessor()                                   # config=..., device="mac"/"cuda"/"cpu", workers=...
finals = pre.process("doc.pdf")                                # list[np.ndarray], page order
pages  = pre.process("doc.pdf", return_all=True)               # list[PageResult]: .page .original .final .applied .files
pre.process("doc.pdf", save=True)                              # save in the current directory
pre.process("doc.pdf", save=True, output_dir="out")            # ... or in out/
for page in pre.iter_pages("big.pdf", save=True, output_dir="out"):   # one page at a time (very large PDFs)
    ...
```
Command line: `python image_processing.py doc.pdf -o out`.

In [ ]:
RUN_API_DEMO = False
if RUN_API_DEMO:
    pre = DocumentPreprocessor()
    pages = pre.process(Path(CONFIG["input"]["pdf_dir"]) / "FIR-2-2009.pdf", save=True, output_dir="output_api",
                        return_all=True)
    print(len(pages), "pages |", pages[0].files, "| needs review:", [p.page for p in pages if p.applied["needs_review"]])

## Tests: information preservation

Planted "content traps", each something a real FIR file can contain, are pasted onto a real clean page. The page then runs through the whole pipeline (Steps 2–10, nothing written), and each trap is checked. A failing test means a step removed information it must keep.

| Trap | Must survive |
|---|---|
| Black banner with **white text** at the top edge (looks like a dark border) | white text still readable on dark (text-vs-banner contrast ≥ 60% of the original) |
| Dark **photo** block 25 × 30 mm (a dark object wider than the background window) | mean tone within 15% and detail (std) ≥ 80% of the original |
| **Faint purple stamp** (grainy dots, light) | ≥ 80% of its coloured pixels still coloured |
| **Faint pencil** strokes (light grey, L ≈ 215) | still visible: ≥ 80% of the stroke pixels darker than 235 |
| **Thin blue line** (1 px) | keeps its colour: chroma ≥ 60% of the original |
| **Light blue writing only** (no dark blue on the page) | not treated as bleed-through by Step 9 |

The pipeline also runs a second time on the same page to confirm the results are reproducible (deterministic).

In [ ]:
# Information-preservation test (code: image_processing.make_trap_page / trap_checks / run_trap_test)
RUN_TESTS = True
if RUN_TESTS:
    base_label = "FIR-4-2013 p4"                              # a clean printed form page
    base = next((r for r in page_records if page_label(r) == base_label),
                next((r for r in page_records if not r["needs"]["skip_page"]), page_records[0]))
    results, same, trap = run_trap_test(base)
    print(f"Information-preservation tests on {page_label(base)} (+ planted traps):")
    for name, (ok, msg) in results.items():
        print(f"  {'PASS' if ok else 'FAIL'}  {name:<24} {msg}")
    print("  flags raised on the trap page:", sorted({f['code'] for f in trap['flags']}))
    print(f"  {'PASS' if same else 'FAIL'}  deterministic             identical output on a second run")
    show_before_after([trap], "original", "contrast", "Test page: planted traps (before | after)", STEP4, max_pages=1, h=900)
    assert all(ok for ok, _ in results.values()) and same, "information-preservation test failed"